# jets_balance1: balanced flow and wave signal order by order

Reads `jets_balance1.cdf` (model) and `jets_balance1_balance.cdf` (the orders 0..4 of the
balance after Warn et al. and the norm of the residual) written by `jets_balance1.py`.
Set `name` and the time index `it`.

In [ ]:
import numpy as np
import pylab as plt
from netCDF4 import Dataset as NF

name = 'jets_balance1'
it   = -1                                   # time index
k    = -5                                   # vertical level, -1 is the surface (w is zero there for the rigid lid)
order_max = 4                               # orders in the file

f2 = NF(name + '_balance.cdf')
x, y, z = f2.variables['xt'][:]/1e3, f2.variables['yt'][:]/1e3, f2.variables['zt'][:]
t = f2.variables['time'][:]/86400.
norm = f2.variables['norm'][:]
print('time [d]', t)
for m in range(order_max + 1):
    print('order %d: norm of the residual' % m, np.array2string(norm[:, m], precision=2))

In [ ]:
plt.figure(figsize=(7, 4))
for m in range(order_max + 1):
    plt.semilogy(t, norm[:, m], 'o-', label='order %d' % m)
plt.xlabel('time [d]'); plt.ylabel('norm of the residual'); plt.legend(); plt.grid()
plt.title('2 |u - u_bal|^2 / (|u|^2 + |u_bal|^2),  u_bal balanced to order n')
plt.show()

In [ ]:
def balanced(var, order):
    return sum(f2.variables['%s_%d' % (var, m)][it] for m in range(order + 1))

order = order_max
fig = plt.figure(figsize=(18, 9))
for row, (var, fac, lab) in enumerate((('w', 1e3, 'w [mm/s]'), ('v', 1., 'v [m/s]'), ('b', 1e3, 'b [1e-3 m/s^2]'))):
    a = f2.variables[var][it]*fac
    a_bal = balanced(var, order)*fac
    for col, (f, title) in enumerate(((a, lab), (a_bal, lab + ' balanced to order %d' % order), (a - a_bal, 'residual'))):
        ax = plt.subplot(3, 3, 3*row + col + 1)
        co = plt.contourf(x, y, f[k], 21, cmap='RdBu_r')
        plt.colorbar(co); plt.title(title)
        if row == 2: plt.xlabel('x [km]')
        if col == 0: plt.ylabel('y [km]')
plt.suptitle('t = %4.1f d, level %d' % (t[it], k))
plt.tight_layout(); plt.show()

In [ ]:
# the residual of w at every order
fig = plt.figure(figsize=(18, 3.5))
w = f2.variables['w'][it]*1e3
for m in range(order_max + 1):
    ax = plt.subplot(1, order_max + 1, m + 1)
    co = plt.contourf(x, y, (w - balanced('w', m)*1e3)[k], 21, cmap='RdBu_r')
    plt.colorbar(co); plt.title('w - w_bal, order %d [mm/s]' % m)
plt.tight_layout(); plt.show()